# MyLLM - Lightning Fast GPU Pretraining (Google Colab)
Trains at **140,000+ tokens/sec** using in-RAM zero-disk tensor loading, Fused AdamW, and FP16 automatic mixed precision.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Pull latest high-speed training and generation code
import os
if not os.path.exists('/content/myllm'):
    !git clone https://github.com/aaravsuthar45-bit/myllm.git /content/myllm
%cd /content/myllm
!git pull

In [ ]:
# 3. Install dependencies
%pip install -q tokenizers tqdm ftfy datasets

### 4. Load Data from Drive or Upload
If your dataset is in Google Drive, mount and sync it to `/content/myllm/data`.

In [ ]:
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# If you saved training_data.jsonl or pretraining/ in Google Drive:
DRIVE_DATA = '/content/drive/MyDrive/MY-LLM/data'
if os.path.exists(DRIVE_DATA):
    !rsync -av "$DRIVE_DATA/" /content/myllm/data/
    print("Synced data from Google Drive to local Colab SSD!")

### 5. Tokenize New Data (Skip if already tokenized)

In [ ]:
if os.path.exists('data/training_data.jsonl') and not os.path.exists('data/pretraining/fineweb_train.bin'):
    !python training/prepare_fineweb_pretrain.py \
        --input data/training_data.jsonl \
        --tokenizer data/tokenizer/tokenizer_combined.json \
        --output_dir data/pretraining \
        --val_fraction 0.05

### 6. High-Speed Training (140,000+ tokens/sec)
Trains for 3,000 steps with live tok/s and ETA. Finishes in ~3 to 5 minutes!

In [ ]:
!python training/pretrain.py \
    --batch_size 64 \
    --max_steps 3000 \
    --eval_every 250 \
    --log_every 50 \
    --learning_rate 6e-4

### 7. Test Generation with Trained MyLLM Checkpoint
Runs inference directly using your trained Transformer model weights.

In [ ]:
!python generate.py \
    --prompt "Artificial intelligence is" \
    --checkpoint checkpoints/pretrained_best.pt \
    --max_tokens 100 \
    --temperature 0.8